In [ ]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../..")

from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aeema_articulatory_features

In [ ]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aeema.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
token_df = dataset_df_to_token_df(df)

In [3]:
scalers = {}
for subject in token_df["subject"].unique():
    with open(f"../encoder_training/aeema_scalers/{subject}.pkl", "rb") as f:
        scalers[subject] = pickle.load(f)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import SequenceAutoencoder

device = "mps" if torch.backends.mps.is_available() else "cpu"

art_model = SequenceAutoencoder(
    feature_shape=(6, 3),
    latent_dim=1024,
).to(device)
art_model.load_state_dict(torch.load("../encoder_training/ckpts/aeema/art_model_ckpt.pth", map_location=device))
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)
ac_model.load_state_dict(torch.load("../encoder_training/ckpts/aeema/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (enc_conv): Sequential(
    (0): Conv1d(30, 512, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(512, 256, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
  )
  (enc_pool): AdaptiveAvgPool1d(output_size=1)
  (fc_enc): Sequential(
    (0): Linear(in_features=128, out_features=4096, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4096, out_features=2048, bias=True)
    (3): ReLU()
    (4): Linear(in_features=2048, out_features=1024, bias=True)
  )
  (fc_dec): Sequential(
    (0): Linear(in_features=1024, out_features=2048, bias=True)
    (1): ReLU()
    (2): Linear(i

In [5]:
for name, param in art_model.named_parameters():
    if not torch.isfinite(param).all():
        print(
            name,
            "NaN:", torch.isnan(param).sum().item(),
            "Inf:", torch.isinf(param).sum().item(),
        )

In [6]:
art_vectors = []
ac_vectors = []
phonemes = []
subjects = []

pbar = tqdm.tqdm(token_df.iterrows(), total=len(token_df))

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]

    articulatory_features = get_aeema_articulatory_features(art_f, start, end, scalers=scalers[subject])
    if articulatory_features is None: continue
    articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 6, 3)
    articulatory_features = torch.from_numpy(articulatory_features).float().unsqueeze(0).to(device)
    art_out, art_z = art_model(articulatory_features)

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is None: continue
    mfccs = torch.from_numpy(mfccs.T).unsqueeze(0).to(device)
    ac_out, ac_z = ac_model(mfccs)

    art_vectors.append(art_z.detach().cpu().numpy().squeeze())
    ac_vectors.append(ac_z.detach().cpu().numpy().squeeze())
    phonemes.append(phone)
    subjects.append(subject)

100%|██████████| 56882/56882 [25:34<00:00, 37.06it/s]


In [7]:
import numpy as np
art = np.stack(art_vectors, axis=0)   # (N, T_A, D_A)
ac = np.stack(ac_vectors, axis=0)  # (N, T_X, D_X)

# A: (N, T_A, D_A)
# X: (N, T_X, D_X)
N, D = art.shape
N2, D_X = ac.shape
assert N == N2

art_flat = art.reshape(N, D)
ac_flat = ac.reshape(N, D_X)   # shape (N, x_dim)

print("art_flat:", art_flat.shape, "ac_flat:", ac_flat.shape)

art_flat: (56818, 1024) ac_flat: (56818, 1024)


In [8]:
art_raw = art_flat.astype(np.float32)
ac_raw = ac_flat.astype(np.float32)

N = len(ac_raw)
k = 5

indices = np.arange(N)
np.random.shuffle(indices)
folds = np.array_split(indices, k)

In [9]:
art_dim = art_raw.shape[1]
ac_dim = ac_raw.shape[1]

In [10]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

# ----------------------------------------------------
# Dataset
# ----------------------------------------------------
class AXDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# ----------------------------------------------------
# Training loop for one model
# ----------------------------------------------------
def train_model(train_loader, val_loader, x_dim, y_dim):
    model = MDN(x_dim=x_dim, y_dim=y_dim, n_components=5, hidden_dim=256).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val = float('inf')
    patience = 10
    counter = 0

    for epoch in range(1, 2001):  # just large upper bound
        # ----- train -----
        model.train()
        train_loss = 0
        for Xb, Yb in train_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            logp = model.log_prob(Xb, Yb)
            loss = -logp.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(Xb)
        train_loss /= len(train_loader.dataset)

        # ----- validation -----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, Yb in val_loader:
                Xb, Yb = Xb.to(device), Yb.to(device)
                loss = -model.log_prob(Xb, Yb).mean()
                val_loss += loss.item() * len(Xb)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

        # Early stopping check
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
            print("   ✓ New best model")
        else:
            counter += 1
            print(f"   No improvement ({counter} epochs)")

        if counter >= patience:
            print(f"Early stopping. Best val={best_val:.4f}")
            break

    model.load_state_dict(best_state)
    return model

In [11]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]


    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_ac = StandardScaler()
    sc_art = StandardScaler()

    sc_ac.fit(ac_raw[train_inner])
    sc_art.fit(art_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    ac_norm = sc_ac.transform(ac_raw)
    art_norm = sc_art.transform(art_raw)

    # Dataset for p(ART | AC)
    dataset = AXDataset(ac_norm, art_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=ac_raw.shape[1],
        y_dim=art_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            ac_fold,
            art_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    ART coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_art.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_art_given_ac_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_art_given_ac_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=692.7499, val=582.4872
   ✓ New best model
Epoch 2: train=561.3255, val=508.4338
   ✓ New best model
Epoch 3: train=523.2464, val=472.9200
   ✓ New best model
Epoch 4: train=482.3846, val=480.2454
   No improvement (1 epochs)
Epoch 5: train=469.7836, val=477.5338
   No improvement (2 epochs)
Epoch 6: train=464.9131, val=474.1538
   No improvement (3 epochs)
Epoch 7: train=464.3611, val=465.7970
   ✓ New best model
Epoch 8: train=465.3361, val=470.4951
   No improvement (1 epochs)
Epoch 9: train=460.9372, val=457.4987
   ✓ New best model
Epoch 10: train=460.9853, val=461.5224
   No improvement (1 epochs)
Epoch 11: train=460.7802, val=480.8436
   No improvement (2 epochs)
Epoch 12: train=458.1897, val=456.5985
   ✓ New best model
Epoch 13: train=459.2037, val=463.7475
   No improvement (1 epochs)
Epoch 14: train=456.5586, val=466.7372
   No improvement (2 epochs)
Epoch 15: train=455.7538, val=458.8609
   No improvement (3 epochs)
Epoch 16: train=453.9287, val=45

In [12]:
import pickle
with open("entropies/aeema_art_given_ac.pickle", "wb") as f:
    pickle.dump(H_art_given_ac_mdn, f)

In [13]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]

    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_art = StandardScaler()
    sc_ac = StandardScaler()

    sc_art.fit(art_raw[train_inner])
    sc_ac.fit(ac_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    art_norm = sc_art.transform(art_raw)
    ac_norm = sc_ac.transform(ac_raw)

    # Dataset for p(AC | ART)
    dataset = AXDataset(art_norm, ac_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=art_raw.shape[1],
        y_dim=ac_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            art_fold,
            ac_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    AC coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_ac.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_ac_given_art_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_ac_given_art_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=114.8718, val=85.4921
   ✓ New best model
Epoch 2: train=0.2188, val=-42.5683
   ✓ New best model
Epoch 3: train=-12.3453, val=-36.5985
   No improvement (1 epochs)
Epoch 4: train=-63.7511, val=-62.9572
   ✓ New best model
Epoch 5: train=-67.1282, val=-42.8968
   No improvement (1 epochs)
Epoch 6: train=-77.4609, val=-94.9687
   ✓ New best model
Epoch 7: train=-101.3799, val=-42.3030
   No improvement (1 epochs)
Epoch 8: train=-94.2434, val=-61.3770
   No improvement (2 epochs)
Epoch 9: train=-102.4818, val=-98.0854
   ✓ New best model
Epoch 10: train=-106.9439, val=-69.8048
   No improvement (1 epochs)
Epoch 11: train=-100.7079, val=-67.1664
   No improvement (2 epochs)
Epoch 12: train=-109.2117, val=-107.7142
   ✓ New best model
Epoch 13: train=-111.4203, val=-95.5901
   No improvement (1 epochs)
Epoch 14: train=-113.3386, val=-114.2436
   ✓ New best model
Epoch 15: train=-122.8184, val=-93.9537
   No improvement (1 epochs)
Epoch 16: train=-118.5930, val=-11

In [14]:
with open("entropies/aeema_ac_given_art.pickle", "wb") as f:
    pickle.dump(H_ac_given_art_mdn, f)